In [0]:
# ============================================================
# CÉLULA 1 — Preparação da camada Gold
# VoeBem Analytics
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Configuração da arquitetura
# ------------------------------------------------------------

CATALOG = "voebem"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

TABELA_SILVER_VRA = f"{CATALOG}.{SCHEMA_SILVER}.vra"

# ------------------------------------------------------------
# Criação do schema Gold
# ------------------------------------------------------------

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}"
)

# ------------------------------------------------------------
# Leitura da camada Silver
# ------------------------------------------------------------

df_vra = spark.table(TABELA_SILVER_VRA)

# ------------------------------------------------------------
# Validação
# ------------------------------------------------------------

print("=== VOEBEM ANALYTICS — CAMADA GOLD ===")
print(f"Origem: {TABELA_SILVER_VRA}")
print(f"Destino: {CATALOG}.{SCHEMA_GOLD}")
print(f"Registros disponíveis: {df_vra.count():,}")
print(f"Colunas disponíveis: {len(df_vra.columns)}")
print("Schema Gold preparado com sucesso.")

In [0]:
# ============================================================
# CÉLULA 2 — Gold: KPIs por companhia aérea
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Agregação dos indicadores por companhia
# ------------------------------------------------------------

df_gold_companhias = (
    df_vra
    .groupBy("icao_empresa_aerea")
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.round(
            F.avg("atraso_chegada_min"), 2
        ).alias("atraso_medio_chegada_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

# ------------------------------------------------------------
# Percentual de voos atrasados
# Proteção explícita contra divisão por zero
# ------------------------------------------------------------

df_gold_companhias = (
    df_gold_companhias
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(F.desc("total_voos"))
)

# ------------------------------------------------------------
# Persistência em Delta
# ------------------------------------------------------------

TABELA_GOLD_COMPANHIAS = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_companhias"
)

(
    df_gold_companhias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_COMPANHIAS)
)

# ------------------------------------------------------------
# Validação
# ------------------------------------------------------------

print(f"Tabela criada: {TABELA_GOLD_COMPANHIAS}")
print(f"Companhias encontradas: {df_gold_companhias.count():,}")

display(df_gold_companhias)

In [0]:
# ============================================================
# CÉLULA 3 — Gold: KPIs por rota
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Agregação dos indicadores por rota
# Origem + Destino
# ------------------------------------------------------------

df_gold_rotas = (
    df_vra
    .groupBy(
        "icao_aerodromo_origem",
        "icao_aerodromo_destino"
    )
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.count("atraso_chegada_min").alias(
            "voos_com_dados_chegada"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.round(
            F.avg("atraso_chegada_min"), 2
        ).alias("atraso_medio_chegada_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

# ------------------------------------------------------------
# Percentual de voos atrasados
# Proteção contra divisão por zero
# ------------------------------------------------------------

df_gold_rotas = (
    df_gold_rotas
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(F.desc("total_voos"))
)

# ------------------------------------------------------------
# Persistência em Delta
# ------------------------------------------------------------

TABELA_GOLD_ROTAS = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_rotas"
)

(
    df_gold_rotas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_ROTAS)
)

# ------------------------------------------------------------
# Validação
# ------------------------------------------------------------

print(f"Tabela criada: {TABELA_GOLD_ROTAS}")
print(f"Rotas encontradas: {df_gold_rotas.count():,}")

display(df_gold_rotas)

In [0]:
# ============================================================
# CÉLULA 4 — Gold: KPIs por aeroporto de origem
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Agregação dos indicadores por aeroporto de origem
# ------------------------------------------------------------

df_gold_aeroportos = (
    df_vra
    .groupBy("icao_aerodromo_origem")
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.round(
            F.avg("atraso_partida_min"), 2
        ).alias("atraso_medio_partida_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

# ------------------------------------------------------------
# Percentual de voos atrasados
# Proteção contra divisão por zero
# ------------------------------------------------------------

df_gold_aeroportos = (
    df_gold_aeroportos
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(F.desc("total_voos"))
)

# ------------------------------------------------------------
# Persistência em Delta
# ------------------------------------------------------------

TABELA_GOLD_AEROPORTOS = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_aeroportos_origem"
)

(
    df_gold_aeroportos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_AEROPORTOS)
)

# ------------------------------------------------------------
# Validação
# ------------------------------------------------------------

print(f"Tabela criada: {TABELA_GOLD_AEROPORTOS}")
print(f"Aeroportos encontrados: {df_gold_aeroportos.count():,}")

display(df_gold_aeroportos)

In [0]:
# ============================================================
# CÉLULA 5 — Gold: KPIs mensais
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# Base temporal
# Utilizamos partida_prevista_ts como referência do voo
# ------------------------------------------------------------

df_gold_mensal = (
    df_vra
    .filter(F.col("partida_prevista_ts").isNotNull())
    .withColumn(
        "ano",
        F.year("partida_prevista_ts")
    )
    .withColumn(
        "mes",
        F.month("partida_prevista_ts")
    )
    .withColumn(
        "ano_mes",
        F.date_format(
            "partida_prevista_ts",
            "yyyy-MM"
        )
    )
    .groupBy(
        "ano",
        "mes",
        "ano_mes"
    )
    .agg(
        F.count("*").alias("total_voos"),

        F.count("atraso_partida_min").alias(
            "voos_com_dados_partida"
        ),

        F.count("atraso_chegada_min").alias(
            "voos_com_dados_chegada"
        ),

        F.round(
            F.avg("atraso_partida_min"),
            2
        ).alias("atraso_medio_partida_min"),

        F.round(
            F.avg("atraso_chegada_min"),
            2
        ).alias("atraso_medio_chegada_min"),

        F.sum(
            F.when(
                F.col("atraso_partida_min") > 15,
                1
            ).otherwise(0)
        ).alias("voos_atrasados_15min")
    )
)

# ------------------------------------------------------------
# Percentual de atrasos
# Protegido contra divisão por zero
# ------------------------------------------------------------

df_gold_mensal = (
    df_gold_mensal
    .withColumn(
        "percentual_atrasados_15min",
        F.when(
            F.col("voos_com_dados_partida") > 0,
            F.round(
                (
                    F.col("voos_atrasados_15min")
                    / F.col("voos_com_dados_partida")
                ) * 100,
                2
            )
        ).otherwise(F.lit(None))
    )
    .orderBy(
        "ano",
        "mes"
    )
)

# ------------------------------------------------------------
# Persistência em Delta
# ------------------------------------------------------------

TABELA_GOLD_MENSAL = (
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_mensal"
)

(
    df_gold_mensal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_MENSAL)
)

# ------------------------------------------------------------
# Validação
# ------------------------------------------------------------

print(f"Tabela criada: {TABELA_GOLD_MENSAL}")
print(f"Períodos encontrados: {df_gold_mensal.count():,}")

display(df_gold_mensal)

In [0]:
# ============================================================
# CÉLULA 6 — Validação final da camada Gold
# ============================================================

tabelas_gold = [
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_companhias",
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_rotas",
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_aeroportos_origem",
    f"{CATALOG}.{SCHEMA_GOLD}.kpi_mensal"
]

print("=" * 70)
print("VALIDAÇÃO FINAL — CAMADA GOLD")
print("=" * 70)

for tabela in tabelas_gold:
    df = spark.table(tabela)

    print(f"\nTabela: {tabela}")
    print(f"Registros: {df.count():,}")
    print(f"Colunas: {len(df.columns)}")

print("\n" + "=" * 70)
print("CAMADA GOLD VALIDADA")
print("=" * 70)